In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

pd.set_option('display.max_columns', 100)

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/FIAP/Machine Learning & Data Science/2026/Datasets/dataset_armageddon_final_corrected.csv")

In [ ]:
df.sample(5)

,peso,potencia,velocidade_max,tipo_cambio,tipo_pneu,tipo_motor,sobrealimentacao,carroceria,tracao,pista,titulos_armageddon,vitorias_lista_estado,tempo_armageddon_s
2100,1613.391870,1849.859474,271.192466,Liberty,Pro-Front,AP,Aspirado,Fibra de Carbono,4x4,Pista Tratada,4,16,6.016483
2735,1742.227452,2385.824419,276.512500,Liberty,Hoosier,L6,Aspirado,Metal,Dianteira,No Prep,5,37,5.306555
2196,1217.186933,1435.011931,271.965774,Sapinho,Mickey Thompson,V8,Supercharger,Fibra de Vidro,Traseira,Pista Tratada,9,9,6.283843
658,1338.835586,2342.577014,285.591543,Sequencial,Hoosier,5C,Single Turbo,Fibra de Carbono,Dianteira,No Prep,8,8,5.248627
1422,764.898688,1553.114708,292.294484,Sapinho,Hoosier,5C,Bi-Turbo,Fibra de Vidro,4x4,Pista Tratada,9,19,5.431185


In [ ]:
cols = df.select_dtypes(include="object") # lista as variáveis tipo objeto (texto, categorica)

for c in cols:
    df[c] = df[c].astype("category") # altera o tipo da variável para categorico
    df[c] = df[c].cat.codes # transforma as categorias em números

#### *Separação entre treino e teste*

In [ ]:
# separação de variáveis dependente e independentes

X = df.drop("tempo_armageddon_s", axis=1) # Armazena todas as variáveis menos a variável alvo
y = df["tempo_armageddon_s"] # armazena apenas a variável alvo

In [ ]:
# dados de treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### *Bagging Models*

In [ ]:
rf_model = RandomForestRegressor() # Instancia o algoritmo
rf_model.fit(X_train, y_train) # Treina o modelo

RandomForestRegressor()

In [ ]:
# teste do modelo
y_pred = rf_model.predict(X_test)

In [ ]:
def avalia_modelo(y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape
            }

In [ ]:
avalia_modelo(y_test, y_pred)

{'MAE': 0.13856418706338594,
 'MSE': 0.03020557686124484,
 'RMSE': np.float64(0.17379751684430028),
 'R2': 0.9463379427991355,
 'MAPE': np.float64(2.5638513766623463)}

In [ ]:
# Extra Tree
from sklearn.ensemble import ExtraTreesRegressor

et = ExtraTreesRegressor()
et.fit(X_train, y_train)

ExtraTreesRegressor()

In [ ]:
y_pred = et.predict(X_test)

In [ ]:
avalia_modelo(y_test, y_pred)

{'MAE': 0.14029089897769617,
 'MSE': 0.03091602010428891,
 'RMSE': np.float64(0.17582952000244131),
 'R2': 0.9450757968675638,
 'MAPE': np.float64(2.5982691497655317)}

#### *Modelos Boosting*

In [ ]:
!pip install xgboost lightgbm catboost --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 5.8 MB/s eta 0:00:00


In [ ]:
from sklearn.ensemble import AdaBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [ ]:
# Loop de treinamento
# Modelos sem ajuste de hiperparâmetros
modelos = {
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "AdaBoost": AdaBoostRegressor(),
    "XGBoost": XGBRegressor(verbosity=0),
    "LightGBM": LGBMRegressor(verbose=-1),
    "CatBoost": CatBoostRegressor(verbose=0),
    "Extra Trees": ExtraTreesRegressor()
}

In [ ]:
for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    print(f"{nome} -  R² = {r2:.2f}")

Random Forest -  R² = 0.95
Gradient Boosting -  R² = 0.95
AdaBoost -  R² = 0.94
XGBoost -  R² = 0.94
LightGBM -  R² = 0.95
CatBoost -  R² = 0.95
Extra Trees -  R² = 0.95
